In [23]:
import earthaccess
auth = earthaccess.login()
print(auth.authenticated)

True


In [24]:
import rasterio
import numpy as np

dem_path = "Sikkim.tif"  

with rasterio.open(dem_path) as dem:
    dem_array = dem.read(1)  # elevation values as a 2D array
    dem_transform = dem.transform
    dem_crs = dem.crs
    dem_bounds = dem.bounds
    dem_nodata = dem.nodata

print("Shape:", dem_array.shape)
print("Bounds:", dem_bounds)
print("CRS:", dem_crs)
print("Nodata value:", dem_nodata)
print("Elevation range:", np.nanmin(dem_array), "to", np.nanmax(dem_array))

Shape: (3285, 2294)
Bounds: BoundingBox(left=88.1815277778133, bottom=27.00486111110674, right=88.8187500000356, top=27.917361111106864)
CRS: EPSG:4326
Nodata value: -32768.0
Elevation range: 183 to 7346


In [25]:
dem_array = np.where(dem_array == dem_nodata, np.nan, dem_array)

In [26]:
from scipy import ndimage

# pixel size in meters (approx, since SRTM is in degrees — adjust for latitude)
pixel_size_x = dem_transform[0] * 111320 * np.cos(np.radians(27.5))  # rough meters/pixel at Sikkim's latitude
pixel_size_y = abs(dem_transform[4]) * 111320

dzdx = ndimage.sobel(dem_array, axis=1) / (8 * pixel_size_x)
dzdy = ndimage.sobel(dem_array, axis=0) / (8 * pixel_size_y)

slope_rad = np.arctan(np.sqrt(dzdx**2 + dzdy**2))
slope_deg = np.degrees(slope_rad)

print("Slope range:", np.nanmin(slope_deg), "to", np.nanmax(slope_deg))
print("Slope 95th percentile:", np.nanpercentile(slope_deg, 95))
print("Slope mean:", np.nanmean(slope_deg))

Slope range: 0.0 to 86.72215997189346
Slope 95th percentile: 49.68170823609534
Slope mean: 28.95248174338637


In [27]:
import earthaccess

results = earthaccess.search_data(
    short_name="SPL3SMP_E",
    bounding_box=(dem_bounds.left, dem_bounds.bottom, dem_bounds.right, dem_bounds.top),
    temporal=("2024-06-01", "2024-06-10")
)
print(len(results))

10


In [28]:
import earthaccess
auth = earthaccess.login()
print(auth.authenticated)

True


In [29]:
import rasterio
import numpy as np
import glob

# grab one soil_moisture file to test
sm_files = sorted(glob.glob("smap_data/*soil_moisture*.tif"))
print("Total soil moisture files:", len(sm_files))
print("First file:", sm_files[0])

with rasterio.open(sm_files[0]) as src:
    sm_array = src.read(1)
    sm_bounds = src.bounds
    sm_crs = src.crs
    sm_transform = src.transform
    sm_nodata = src.nodata

print("Shape:", sm_array.shape)
print("Bounds:", sm_bounds)
print("CRS:", sm_crs)
print("Nodata:", sm_nodata)
print("Value range:", np.nanmin(sm_array), "to", np.nanmax(sm_array))

Total soil moisture files: 383
First file: smap_data/SPL3SMP_E.006_Soil_Moisture_Retrieval_Data_AM_soil_moisture_20220101T000000_aid0001.tif
Shape: (12, 9)
Bounds: BoundingBox(left=88.15610219843427, bottom=26.97846511923281, right=88.87023803982572, top=27.930646241088084)
CRS: EPSG:4326
Nodata: -9999.0
Value range: -9999.0 to 0.36533302


In [30]:
import netrc
import requests

# read credentials from .netrc automatically
secrets = netrc.netrc()
username, _, password = secrets.authenticators("urs.earthdata.nasa.gov")

# Step 1: Login to get a token
login_resp = requests.post(
    "https://appeears.earthdatacloud.nasa.gov/api/login",
    auth=(username, password)
)
print(login_resp.status_code)
print(login_resp.json())

200
{'token_type': 'Bearer', 'token': 'fpRBS5r8O3JtQB5_5G3k7MTfv0IIaUt4tVErCCbQWkh3W0B13LRW3kev7qXIq74CkfvuKbDBXTS0dMu_lsh1UA', 'expiration': '2026-09-10T13:47:27Z'}


In [31]:
token = login_resp.json()["token"]

headers = {"Authorization": f"Bearer {token}"}
task_id = "4247fc3c-ee4b-4e75-95b3-c59354eafc9c"
bundle_url = f"https://appeears.earthdatacloud.nasa.gov/api/bundle/{task_id}"

response = requests.get(bundle_url, headers=headers)
bundle = response.json()
files = bundle["files"]
print("Total files in bundle:", len(files))

Total files in bundle: 1440


In [32]:
soil_moisture_files = [f for f in files if "soil_moisture" in f["file_name"] and "qual_flag" not in f["file_name"]]
qual_flag_files = [f for f in files if "qual_flag" in f["file_name"]]
other_files = [f for f in files if "soil_moisture" not in f["file_name"] and "qual_flag" not in f["file_name"]]

print("Soil moisture files:", len(soil_moisture_files))
print("Qual flag files:", len(qual_flag_files))
print("Other/supporting files:", len(other_files))

Soil moisture files: 383
Qual flag files: 1050
Other/supporting files: 7


In [33]:
import re

dates = sorted([re.search(r'(\d{8})T000000', f["file_name"]).group(1) for f in soil_moisture_files])
print("Earliest:", dates[0])
print("Latest:", dates[-1])
print("Total unique dates:", len(set(dates)))

Earliest: 20220101
Latest: 20241230
Total unique dates: 383


In [34]:
import os
import rasterio
import numpy as np

def load_soil_moisture(filepath):
    with rasterio.open(filepath) as src:
        arr = src.read(1)
        nodata = src.nodata
        arr = np.where(arr == nodata, np.nan, arr)
    return arr

# Extract base filename and point to the local smap_data/ folder
filename = os.path.basename(soil_moisture_files[0]["file_name"])
local_path = os.path.join("smap_data", filename)

# quick test
test_arr = load_soil_moisture(local_path)
print(test_arr.shape, np.nanmin(test_arr), np.nanmax(test_arr))


(12, 9) 0.1273873 0.36533302


In [35]:
from rasterio.warp import reproject, Resampling

with rasterio.open(local_path) as sm_src:
    sm_array = sm_src.read(1)
    sm_transform = sm_src.transform
    sm_crs = sm_src.crs
    sm_nodata = sm_src.nodata

sm_array = np.where(sm_array == sm_nodata, np.nan, sm_array)

sm_resampled = np.empty(dem_array.shape, dtype=np.float32)

reproject(
    source=sm_array,
    destination=sm_resampled,
    src_transform=sm_transform,
    src_crs=sm_crs,
    dst_transform=dem_transform,
    dst_crs=dem_crs,
    resampling=Resampling.bilinear
)

print("Resampled shape:", sm_resampled.shape)
print("Should match DEM shape:", dem_array.shape)
print("Value range after resample:", np.nanmin(sm_resampled), np.nanmax(sm_resampled))

Resampled shape: (3285, 2294)
Should match DEM shape: (3285, 2294)
Value range after resample: 0.13012834 0.36530203


In [36]:
def resample_soil_moisture(filepath, dem_array, dem_transform, dem_crs):
    with rasterio.open(filepath) as sm_src:
        sm_array = sm_src.read(1)
        sm_transform = sm_src.transform
        sm_crs = sm_src.crs
        sm_nodata = sm_src.nodata

    sm_array = np.where(sm_array == sm_nodata, np.nan, sm_array)

    sm_resampled = np.empty(dem_array.shape, dtype=np.float32)
    reproject(
        source=sm_array,
        destination=sm_resampled,
        src_transform=sm_transform,
        src_crs=sm_crs,
        dst_transform=dem_transform,
        dst_crs=dem_crs,
        resampling=Resampling.bilinear
    )
    return sm_resampled

In [37]:
soil_moisture_stack = {}

for f in soil_moisture_files:
    filename = os.path.basename(f["file_name"])
    date_str = re.search(r'(\d{8})T000000', filename).group(1)
    local_path = os.path.join("smap_data", filename)
    
    soil_moisture_stack[date_str] = resample_soil_moisture(local_path, dem_array, dem_transform, dem_crs)

print("Total dates processed:", len(soil_moisture_stack))

Total dates processed: 383
